In [2]:
import openmeteo_requests
import polars as pl
import requests_cache
import os
from retry_requests import retry
from datetime import datetime, timedelta, timezone


# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Get location information
locations_df = pl.read_csv("locations.csv")

location_names = locations_df["name"].to_list()
latitudes = locations_df["lat"].to_list()
longitudes = locations_df["lon"].to_list()

# URL and parameters remain the same
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude":latitudes,
    "longitude": longitudes,
    "start_date": "2021-01-01",
    "end_date": "2025-12-31",
    "daily": ["weather_code", "temperature_2m_mean", "precipitation_sum"],
    "hourly": ["temperature_2m", "precipitation", "snow_depth", "snowfall", "wind_speed_10m", "wind_gusts_10m", "relative_humidity_2m", "weather_code"],
}
responses = openmeteo.weather_api(url, params=params)
print("finished")



finished


In [3]:
# Hourly Data
BASE_DIR   = "../../data"
OUTPUT_DIR = os.path.join(BASE_DIR, "Weather", "weather_hourly.csv")

all_dataframes = []
# Process first location
for i, response in enumerate(responses):
    print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevation: {response.Elevation()} m asl")
    print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
    # Get name
    name = location_names[i]
    
    # --- Process Hourly Data ---
    hourly = response.Hourly()
    
    #  calculate the range using datetime objects
    start_h = datetime.fromtimestamp(hourly.Time(), tz=timezone.utc)
    end_h = datetime.fromtimestamp(hourly.TimeEnd(), tz=timezone.utc)
    
    interval_h = timedelta(seconds=hourly.Interval())
    
    hourly_dataframe = pl.DataFrame({
        "date": pl.datetime_range(start_h, end_h - interval_h, interval_h, eager=True),
        "location_name": name,
        "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
        "precipitation": hourly.Variables(1).ValuesAsNumpy(),
        "snow_depth": hourly.Variables(2).ValuesAsNumpy(),
        "snowfall": hourly.Variables(3).ValuesAsNumpy(),
        "wind_speed_10m": hourly.Variables(4).ValuesAsNumpy(),
        "wind_gusts_10m": hourly.Variables(5).ValuesAsNumpy(),
        "relative_humidity_2m": hourly.Variables(6).ValuesAsNumpy(),
        "weather_code": hourly.Variables(7).ValuesAsNumpy(),
    })
    all_dataframes.append(hourly_dataframe)

final_dataframe = pl.concat(all_dataframes)

final_dataframe.write_csv(
    OUTPUT_DIR,
    datetime_format="%d.%m.%Y %H:%M"
)

print("\nHourly data\n", hourly_dataframe)



Coordinates: 54.3057975769043°N 10.1953125°E
Elevation: 15.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
 shape: (43_824, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ date      ┆ temperatu ┆ precipita ┆ snow_dept ┆ … ┆ wind_spee ┆ wind_gust ┆ relative_ ┆ weather_ │
│ ---       ┆ re_2m     ┆ tion      ┆ h         ┆   ┆ d_10m     ┆ s_10m     ┆ humidity_ ┆ code     │
│ datetime[ ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ 2m        ┆ ---      │
│ μs, UTC]  ┆ f32       ┆ f32       ┆ f32       ┆   ┆ f32       ┆ f32       ┆ ---       ┆ f32      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ f32       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2021-01-0 ┆ 2.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 14.88242  ┆ 32.039997 ┆ 96.143799 ┆ 3.0      │
│ 1         ┆           ┆           ┆           ┆   ┆      

In [17]:
# Daily Data
BASE_DIR   = "../../data"
OUTPUT_DIR = os.path.join(BASE_DIR, "Weather", "weather_daily.csv")

# --- Process Daily Data ---
daily = response.Daily()

start_d = datetime.fromtimestamp(daily.Time(), tz=timezone.utc)
end_d = datetime.fromtimestamp(daily.TimeEnd(), tz=timezone.utc)
interval_d = timedelta(seconds=daily.Interval())

daily_dataframe = pl.DataFrame({
    "date": pl.datetime_range(start_d, end_d - interval_d, interval_d, eager=True),
    "weather_code": daily.Variables(0).ValuesAsNumpy(),
    "temperature_2m_mean": daily.Variables(1).ValuesAsNumpy(),
    "precipitation_sum": daily.Variables(2).ValuesAsNumpy(),
})

daily_dataframe.write_csv(
    OUTPUT_DIR,
    datetime_format="%d.%m.%Y %H:%M"
)
print("\nDaily data\n", daily_dataframe)


Daily data
 shape: (1_826, 4)
┌─────────────────────────┬──────────────┬─────────────────────┬───────────────────┐
│ date                    ┆ weather_code ┆ temperature_2m_mean ┆ precipitation_sum │
│ ---                     ┆ ---          ┆ ---                 ┆ ---               │
│ datetime[μs, UTC]       ┆ f32          ┆ f32                 ┆ f32               │
╞═════════════════════════╪══════════════╪═════════════════════╪═══════════════════╡
│ 2021-01-01 00:00:00 UTC ┆ 53.0         ┆ 2.110417            ┆ 2.9               │
│ 2021-01-02 00:00:00 UTC ┆ 53.0         ┆ 2.8375              ┆ 4.299999          │
│ 2021-01-03 00:00:00 UTC ┆ 51.0         ┆ 2.7125              ┆ 0.5               │
│ 2021-01-04 00:00:00 UTC ┆ 51.0         ┆ 1.995833            ┆ 1.4               │
│ 2021-01-05 00:00:00 UTC ┆ 53.0         ┆ 2.1125              ┆ 3.3               │
│ …                       ┆ …            ┆ …                   ┆ …                 │
│ 2025-12-27 00:00:00 UTC ┆ 3.0   